# 21 — Re-fit All BERTopic Runs

Regenerates **all 20 checkpoints** (coast bands + year slices × en/vi) from scratch.

| Parameter | English | Vietnamese |
|---|---|---|
| `min_cluster_size` | 5 | **10** |
| `min_samples` | 3 | **5** |
| `n_neighbors` (UMAP) | 10 | 10 |

Pipeline per run:
1. Fit BERTopic
2. `reduce_topics(nr_topics=50)` → merge only if raw topics > 50
3. Save checkpoint + write to DuckDB

**Wipes all existing checkpoints and DuckDB rows before starting.**

In [ ]:
import sys
sys.path.insert(0, "..")

import math
import pickle
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from stopwordsiso import stopwords
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer

from src.topic_modeling import load_from_duckdb

# ── Paths ─────────────────────────────────────────────────────────────────
DB_PATH  = Path("../data/hotel_reviews.db")
CKPT_DIR = Path("../checkpoints")
CSV_DIR  = Path("../data/topic_results")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

# ── Run definitions ───────────────────────────────────────────────────────
LANGUAGES = ["en", "vi"]

# Coast bands (km to coastline): A = beachfront, B = near-coast, C = inland.
# Updated 2026-06: B widened to 0.1–1.0 km, C is now ≥ 1.0 km (was 0.1–0.5 / ≥0.5).
BAND_SLICES = {
    "coast_band_A": "r.distance2coastline < 0.1",
    "coast_band_B": "r.distance2coastline >= 0.1 AND r.distance2coastline < 1.0",
    "coast_band_C": "r.distance2coastline >= 1.0",
}

YEAR_SLICES = {f"year_{y}": y for y in range(2018, 2025)}

# ── Per-language cluster params ────────────────────────────────────────────
LANG_PARAMS = {
    "en": {"min_cluster_size": 5,  "min_samples": 3},
    "vi": {"min_cluster_size": 10, "min_samples": 5},
}

# Only merge topics if the raw count exceeds this — otherwise keep original
MAX_TOPICS = 50

# ── Encoder — loaded once, shared across all runs ─────────────────────────
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

print(f"DB          : {DB_PATH.resolve()}")
print(f"Checkpoints : {CKPT_DIR.resolve()}")
print(f"Total runs  : {(len(BAND_SLICES) + len(YEAR_SLICES)) * len(LANGUAGES)}")
print(f"Params EN   : {LANG_PARAMS['en']}")
print(f"Params VI   : {LANG_PARAMS['vi']}")
print(f"reduce_topics only if topics > {MAX_TOPICS}")

## Section 1 — Wipe All Existing Checkpoints & DuckDB Rows

In [2]:
all_run_ids = (
    [f"{base}_{lang}" for base in BAND_SLICES for lang in LANGUAGES]
    + [f"{base}_{lang}" for base in YEAR_SLICES for lang in LANGUAGES]
)

# Delete pkl checkpoints
deleted = []
for run_id in all_run_ids:
    ckpt = CKPT_DIR / f"{run_id}.pkl"
    if ckpt.exists():
        ckpt.unlink()
        deleted.append(ckpt.name)

print(f"Deleted {len(deleted)} checkpoint(s)")
for name in deleted:
    print(f"  {name}")

# Clear DuckDB rows
placeholders = ", ".join(f"'{r}'" for r in all_run_ids)
con = duckdb.connect(str(DB_PATH))
for tbl in ("TOPIC_LABELS", "REVIEW_TOPICS", "TOPIC_ASPECTS"):
    try:
        con.execute(f"DELETE FROM {tbl} WHERE run_id IN ({placeholders})")
        print(f"Cleared {tbl}")
    except Exception as e:
        print(f"  {tbl}: {e}")
con.close()

print(f"\nReady to re-fit {len(all_run_ids)} runs.")

Deleted 8 checkpoint(s)
  year_2018_en.pkl
  year_2018_vi.pkl
  year_2019_en.pkl
  year_2019_vi.pkl
  year_2020_en.pkl
  year_2020_vi.pkl
  year_2021_en.pkl
  year_2021_vi.pkl
Cleared TOPIC_LABELS
Cleared REVIEW_TOPICS
Cleared TOPIC_ASPECTS

Ready to re-fit 20 runs.


## Section 2 — Helpers

In [3]:
def build_topic_model(lang: str) -> BERTopic:
    p = LANG_PARAMS[lang]
    return BERTopic(
        embedding_model=encoder,
        umap_model=UMAP(
            n_neighbors=10,
            n_components=5,
            min_dist=0.0,
            metric="cosine",
            random_state=42,
        ),
        hdbscan_model=HDBSCAN(
            min_cluster_size=p["min_cluster_size"],
            min_samples=p["min_samples"],
            metric="euclidean",
            cluster_selection_method="eom",
            prediction_data=False,
        ),
        vectorizer_model=CountVectorizer(
            stop_words=list(stopwords(["vi", "en"])),
            min_df=2,
            ngram_range=(1, 2),
        ),
        ctfidf_model=ClassTfidfTransformer(),
        representation_model={
            "KeyBERT": KeyBERTInspired(),
            "MMR":     MaximalMarginalRelevance(diversity=0.3),
        },
        nr_topics="auto",
        min_topic_size=p["min_cluster_size"],
        top_n_words=100,
        calculate_probabilities=False,
        verbose=False,
    )


def run_one(run_id: str, docs: list, embeddings, df: pd.DataFrame, lang: str):
    p = LANG_PARAMS[lang]
    print(f"  min_cluster_size={p['min_cluster_size']}  min_samples={p['min_samples']}  docs={len(docs):,}")

    # 1. Fit
    topic_model = build_topic_model(lang)
    topics, _   = topic_model.fit_transform(docs, embeddings)

    n_raw = len(set(topics)) - (1 if -1 in topics else 0)
    n_out = sum(1 for t in topics if t == -1)
    print(f"  Raw: {n_raw} topics  outlier={n_out} ({n_out/len(topics)*100:.1f}%)")

    # 2. Merge only if topics exceed MAX_TOPICS — otherwise keep original
    if n_raw > MAX_TOPICS:
        topic_model.reduce_topics(docs, nr_topics=MAX_TOPICS)
        topics = topic_model.topics_
        n_final = len(set(topics)) - (1 if -1 in topics else 0)
        print(f"  reduce_topics({MAX_TOPICS}): {n_raw} → {n_final} topics")
    else:
        n_final = n_raw
        print(f"  Kept original {n_final} topics (≤ {MAX_TOPICS} — no merge needed)")

    n_out = sum(1 for t in topics if t == -1)
    print(f"  FINAL: {n_final} topics  outlier={n_out} ({n_out/len(topics)*100:.1f}%)")

    # 3. Save checkpoint
    ckpt = CKPT_DIR / f"{run_id}.pkl"
    with open(ckpt, "wb") as f:
        pickle.dump({"model": topic_model, "topics": topics, "language": lang}, f)

    # 4. Write to DuckDB
    write_to_duckdb(run_id, topic_model, df, topics)
    export_csv(run_id, topic_model, df, topics)

    return n_final


def write_to_duckdb(run_id: str, topic_model, df: pd.DataFrame, topics: list):
    con = duckdb.connect(str(DB_PATH))

    topic_info = topic_model.get_topic_info()
    label_rows = []
    for _, row in topic_info.iterrows():
        tid       = int(row["Topic"])
        kw_list   = topic_model.get_topic(tid)
        top_words = ", ".join(w for w, _ in kw_list[:100]) if kw_list else ""
        label_rows.append((run_id, tid, top_words, int(row["Count"]), None, None))

    con.executemany(
        "INSERT OR REPLACE INTO TOPIC_LABELS "
        "(run_id, topic_id, top_words, n_docs, seed_topic, seed_score) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        label_rows,
    )
    con.executemany(
        "INSERT OR REPLACE INTO REVIEW_TOPICS (run_id, review_id, topic_id, prob) "
        "VALUES (?, ?, ?, ?)",
        [(run_id, rid, int(tid), None) for rid, tid in zip(df["review_id"].tolist(), topics)],
    )
    con.close()


def export_csv(run_id: str, topic_model, df: pd.DataFrame, topics: list):
    CSV_DIR.mkdir(parents=True, exist_ok=True)
    topic_model.get_topic_info().assign(run_id=run_id).to_csv(
        CSV_DIR / f"{run_id}_topic_info.csv", index=False, encoding="utf-8-sig"
    )
    result = df[["review_id", "hotel_id", "review_year", "language", "distance2coastline"]].copy()
    result["topic_id"] = topics
    result.insert(0, "run_id", run_id)
    result.to_csv(CSV_DIR / f"{run_id}_review_topics.csv", index=False, encoding="utf-8-sig")


print("Helpers ready.")

Helpers ready.


## Section 3 — Coast Band Runs (A / B / C × en / vi)

In [4]:
coast_summary = []

for run_id_base, where_clause in BAND_SLICES.items():
    for lang in LANGUAGES:
        run_id = f"{run_id_base}_{lang}"
        print(f"\n{'='*60}")
        print(f"Run: {run_id}")

        df, docs, embeddings = load_from_duckdb(
            db_path=DB_PATH,
            language=lang,
            extra_where=f"{where_clause} AND r.distance2coastline IS NOT NULL",
        )

        n_final = run_one(run_id, docs, embeddings, df, lang)
        coast_summary.append({"run_id": run_id, "final_topics": n_final, "n_docs": len(docs)})

print("\n✓ Coast-band runs complete.")
pd.DataFrame(coast_summary)


Run: coast_band_A_en
[load_from_duckdb] Loading 8,719 rows …
[load_from_duckdb] docs: 8,719  embeddings: (8719, 768)
  min_cluster_size=5  min_samples=3  docs=8,719
  Raw: 66 topics  outlier=3522 (40.4%)
  reduce_topics(50): 66 → 49 topics
  FINAL: 49 topics  outlier=3522 (40.4%)

Run: coast_band_A_vi
[load_from_duckdb] Loading 12,170 rows …
[load_from_duckdb] docs: 12,170  embeddings: (12170, 768)
  min_cluster_size=10  min_samples=5  docs=12,170
  Raw: 142 topics  outlier=5641 (46.4%)
  reduce_topics(50): 142 → 49 topics
  FINAL: 49 topics  outlier=5641 (46.4%)

Run: coast_band_B_en
[load_from_duckdb] Loading 15,718 rows …
[load_from_duckdb] docs: 15,718  embeddings: (15718, 768)
  min_cluster_size=5  min_samples=3  docs=15,718
  Raw: 21 topics  outlier=7037 (44.8%)
  Kept original 21 topics (≤ 50 — no merge needed)
  FINAL: 21 topics  outlier=7037 (44.8%)

Run: coast_band_B_vi
[load_from_duckdb] Loading 19,867 rows …
[load_from_duckdb] docs: 19,867  embeddings: (19867, 768)
  min_c

,run_id,final_topics,n_docs
0,coast_band_A_en,49,8719
1,coast_band_A_vi,49,12170
2,coast_band_B_en,21,15718
3,coast_band_B_vi,49,19867
4,coast_band_C_en,49,95033
5,coast_band_C_vi,42,73599


## Section 4 — Year Slice Runs (2018–2024 × en / vi)

In [5]:
year_summary = []

for run_id_base, year in YEAR_SLICES.items():
    for lang in LANGUAGES:
        run_id = f"{run_id_base}_{lang}"
        print(f"\n{'='*60}")
        print(f"Run: {run_id}  (year={year})")

        df, docs, embeddings = load_from_duckdb(
            db_path=DB_PATH,
            language=lang,
            min_year=year,
            max_year=year,
        )

        n_final = run_one(run_id, docs, embeddings, df, lang)
        year_summary.append({"run_id": run_id, "year": year, "lang": lang,
                              "final_topics": n_final, "n_docs": len(docs)})

print("\n✓ Year-slice runs complete.")
pd.DataFrame(year_summary)


Run: year_2018_en  (year=2018)
[load_from_duckdb] Loading 9,362 rows …
[load_from_duckdb] docs: 9,362  embeddings: (9362, 768)
  min_cluster_size=5  min_samples=3  docs=9,362
  Raw: 111 topics  outlier=3300 (35.2%)
  reduce_topics(50): 111 → 49 topics
  FINAL: 49 topics  outlier=3300 (35.2%)

Run: year_2018_vi  (year=2018)
[load_from_duckdb] Loading 4,622 rows …
[load_from_duckdb] docs: 4,622  embeddings: (4622, 768)
  min_cluster_size=10  min_samples=5  docs=4,622
  Raw: 61 topics  outlier=1854 (40.1%)
  reduce_topics(50): 61 → 49 topics
  FINAL: 49 topics  outlier=1854 (40.1%)

Run: year_2019_en  (year=2019)
[load_from_duckdb] Loading 10,986 rows …
[load_from_duckdb] docs: 10,986  embeddings: (10986, 768)
  min_cluster_size=5  min_samples=3  docs=10,986
  Raw: 131 topics  outlier=4248 (38.7%)
  reduce_topics(50): 131 → 49 topics
  FINAL: 49 topics  outlier=4248 (38.7%)

Run: year_2019_vi  (year=2019)
[load_from_duckdb] Loading 6,625 rows …
[load_from_duckdb] docs: 6,625  embeddings:

,run_id,year,lang,final_topics,n_docs
0,year_2018_en,2018,en,49,9362
1,year_2018_vi,2018,vi,49,4622
2,year_2019_en,2019,en,49,10986
3,year_2019_vi,2019,vi,32,6625
4,year_2020_en,2020,en,49,5273
5,year_2020_vi,2020,vi,16,7598
6,year_2021_en,2021,en,25,926
7,year_2021_vi,2021,vi,46,2796
8,year_2022_en,2022,en,49,6905
9,year_2022_vi,2022,vi,49,7796


## Section 5 — Verify

In [6]:
# ── Checkpoint inventory ───────────────────────────────────────────────────
print("=== Checkpoints ===")
for run_id in all_run_ids:
    ckpt = CKPT_DIR / f"{run_id}.pkl"
    status = f"{ckpt.stat().st_size / 1e6:.1f} MB" if ckpt.exists() else "MISSING"
    print(f"  {run_id:<35} {status}")

# ── DuckDB summary ─────────────────────────────────────────────────────────
print("\n=== DuckDB TOPIC_LABELS ===")
con = duckdb.connect(str(DB_PATH), read_only=True)
summary = con.execute("""
    SELECT run_id,
           COUNT(*) FILTER (WHERE topic_id != -1) AS n_topics,
           SUM(n_docs) FILTER (WHERE topic_id != -1) AS docs_in_topics,
           SUM(n_docs) FILTER (WHERE topic_id  = -1) AS outliers
    FROM TOPIC_LABELS
    GROUP BY run_id
    ORDER BY run_id
""").df()
con.close()

print(summary.to_string(index=False))

=== Checkpoints ===
  coast_band_A_en                     1222.0 MB
  coast_band_A_vi                     1260.0 MB
  coast_band_B_en                     1297.0 MB
  coast_band_B_vi                     1342.0 MB
  coast_band_C_en                     2150.2 MB
  coast_band_C_vi                     1919.0 MB
  year_2018_en                        1229.0 MB
  year_2018_vi                        1178.2 MB
  year_2019_en                        1246.5 MB
  year_2019_vi                        1199.5 MB
  year_2020_en                        1184.9 MB
  year_2020_vi                        1209.6 MB
  year_2021_en                        1130.9 MB
  year_2021_vi                        1137.8 MB
  year_2022_en                        1202.5 MB
  year_2022_vi                        1212.3 MB
  year_2023_en                        1290.5 MB
  year_2023_vi                        1280.9 MB
  year_2024_en                        1231.1 MB
  year_2024_vi                        1237.5 MB

=== DuckDB TOPIC_LA